In [2]:
!pip install -q kaggle


In [3]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score


In [4]:
from google.colab import files
files.upload()


Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"danoskario","key":"098675bfbfa6a7791ff7c0a4093bdcca"}'}

In [5]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json


In [6]:
!kaggle competitions download -c ieee-fraud-detection


 77% 91.0M/118M [00:00<00:00, 907MB/s]
100% 118M/118M [00:00<00:00, 553MB/s] 


In [7]:
!unzip ieee-fraud-detection.zip


Archive:  ieee-fraud-detection.zip
  inflating: sample_submission.csv   
  inflating: test_identity.csv       
  inflating: test_transaction.csv    
  inflating: train_identity.csv      
  inflating: train_transaction.csv   


In [8]:
train_trans = pd.read_csv("train_transaction.csv")
train_id = pd.read_csv("train_identity.csv")

# Merge on TransactionID
train_data = train_trans.merge(train_id, on="TransactionID", how="left")

print(train_data.shape)
train_data.head()


(590540, 434)


,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,...,id_31,id_32,id_33,id_34,id_35,id_36,id_37,id_38,DeviceType,DeviceInfo
0,2987000,0,86400,68.5,W,13926,NaN,150.0,discover,142.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2987001,0,86401,29.0,W,2755,404.0,150.0,mastercard,102.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2987002,0,86469,59.0,W,4663,490.0,150.0,visa,166.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2987003,0,86499,50.0,W,18132,567.0,150.0,mastercard,117.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2987004,0,86506,50.0,H,4497,514.0,150.0,mastercard,102.0,...,samsung browser 6.2,32.0,2220x1080,match_status:2,T,F,T,T,mobile,SAMSUNG SM-G892A Build/NRD90M


In [9]:
# Sample 10,000 rows for safe SVM training
train_sample = train_data.sample(n=10000, random_state=42)

print(train_sample['isFraud'].value_counts())


isFraud
0    9649
1     351
Name: count, dtype: int64


In [10]:
drop_cols = ['TransactionID', 'TransactionDT']
X = train_sample.drop(columns=['isFraud'] + drop_cols)
y = train_sample['isFraud']


In [11]:
categorical_cols = X.select_dtypes(include='object').columns

for col in categorical_cols:
    le = LabelEncoder()
    X[col] = X[col].fillna("NA")
    X[col] = le.fit_transform(X[col])


In [12]:
X = X.fillna(0)


In [13]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(" Scaling completed")


✅ Scaling completed


In [ ]:
svm_model = SVC(
    kernel='rbf',
    class_weight='balanced',
    probability=True
)

print("🚀 Training SVM...")
svm_model.fit(X_train_scaled, y_train)

print(" SVM training completed!")


🚀 Training SVM...
✅ SVM training completed!


In [16]:
y_pred = svm_model.predict(X_test_scaled)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))


Accuracy: 0.899

Classification Report:

              precision    recall  f1-score   support

           0       0.97      0.92      0.95      1930
           1       0.12      0.31      0.18        70

    accuracy                           0.90      2000
   macro avg       0.55      0.62      0.56      2000
weighted avg       0.94      0.90      0.92      2000



In [17]:
cm = confusion_matrix(y_test, y_pred)
cm


array([[1776,  154],
       [  48,   22]])

In [ ]:
import joblib

joblib.dump(svm_model, "svm_fraud_model.pkl")
joblib.dump(scaler, "scaler.pkl")

print("Model and scaler saved successfully")


✅ Model and scaler saved successfully


In [19]:
from google.colab import files

files.download("svm_fraud_model.pkl")
files.download("scaler.pkl")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>